# Session 1 · Feedback that holds one drone

By the end of today, the PID you wrote lifts a drone from 0.5 m to 1.5 m and holds it there.

**How it works**: listen to the explanation, type the code with me in the cells marked `✍️`, and run it right away (`Shift+Enter`) to check the plot.
Blanks are `___`. If you get stuck, look at your neighbour's screen or raise your hand. Each next-stage cell already holds the previous stage's code,
so if you miss one, you can rejoin at the next stage.

| Stage | Content |
|---|---|
| 0 | Setup: start the simulator |
| 1 | Hover thrust: how to hang still |
| 2 | Feedback and P: push by the error |
| 3 | D: the brake |
| 4 | I: make up the missing force |
| 5 | Pause the integral when far |
| 6 | Mission: pass on both drones with my gains |
| 7 | Live flight: change values while it flies, PlotJuggler |

## 0. Setup

Run the cell below. The first time takes a few seconds while the physics engine gets ready.
Today's drone is a palm-sized, Crazyflie-class drone, and it only moves up and down (z). Keeping it level is done by the background code.

In [ ]:
from background import *
set_language("en")

warmup()
drone_info()

## 1. Hover thrust: how to hang still

Only two vertical forces act on the drone: the **thrust T** from the motors and the **weight mg**.

$$m\ddot z = T - mg \quad\Rightarrow\quad \text{to hang still: } T = mg$$

✍️ Complete the function that returns the thrust needed to hover. Use the argument `g` instead of 9.81.

In [ ]:
def hover_thrust(mass, g=G):
    return ___  # ✍️


fly_open_loop(hover_thrust)

**Look and think**
- The blue line (exact mass) stays put.
- What if the real drone is 10% heavier? 10% lighter?
- Batteries, markers and wind always make the real drone a little different. Trusting a fixed force (open loop) cannot hold it.
  **We must watch the height and correct.** That is feedback.

## 2. Feedback and P control: push by the error

100 times a second we do this:

1. **Watch**: the height now, `z`
2. **Compare**: the error, `error = target - now` (+ when it must go up)
3. **Correct**: from the error choose a desired acceleration `acc` and add it to the hover thrust: `T = hover + m·acc`

The simplest rule is to push in proportion to the error: **P control**, `acc = kp × error`.
It is a spring: the farther you stretch it, the harder it pulls. `kp` is the spring's stiffness.

✍️ Fill in three places: the P output, the error, and the thrust.
The thrust is $T = \hat m\,(g + u) = \hat m g + \hat m u$ written as code.
$\hat m g$ is the hover thrust from stage 1 (the mass the controller knows is `MASS`), and $u$ is the desired acceleration `acc` from the line just above.

In [ ]:
class PID:
    def __init__(self, kp, ki=0.0, kd=0.0, i_band=None):
        self.kp, self.ki, self.kd = kp, ki, kd
        self.i_band = i_band
        self.integral = 0.0
        self.prev_error = None

    def update(self, error, dt):
        return ___  # ✍️


def altitude_thrust(pid, z_target, z, dt):
    error = ___  # ✍️
    acc = pid.update(error, dt)         # desired upward acceleration [m/s²]
    return ___  # ✍️ thrust T [N] = hover thrust + mass × desired acceleration


step_test(PID(kp=8), altitude_thrust)

**Look and think**: at 4 s the target changes to 1.5 m.
- Does it stop at the target? It bounces like a spring with no brake.
- The middle of the bouncing sits below the target. To hold up the extra 10% of weight an error must remain, so it **sags**.

Change `kp` in the cell below. Weak? Strong?

In [ ]:
compare({"kp = 2": PID(kp=2), "kp = 8": PID(kp=8), "kp = 20": PID(kp=20)}, altitude_thrust)

## 3. D control: the brake

Like braking early before a red light: **if the error is shrinking fast, push less early.**
How fast the error changes is `(current error - previous error) / dt`.

$$acc = k_p\,e + k_d\,\frac{e - e_{\text{prev}}}{dt}$$

On the first call there is no previous error, so use 0.

✍️ Compute the derivative (the rate) and add D to the output.

In [ ]:
class PID:
    def __init__(self, kp, ki=0.0, kd=0.0, i_band=None):
        self.kp, self.ki, self.kd = kp, ki, kd
        self.i_band = i_band
        self.integral = 0.0
        self.prev_error = None

    def update(self, error, dt):
        if self.prev_error is None:
            derivative = ___  # ✍️ first call
        else:
            derivative = ___  # ✍️ (current error - previous error) / dt
        self.prev_error = error
        return ___  # ✍️


compare({"P (kp 8)": PID(kp=8), "PD (kp 8, kd 4)": PID(kp=8, kd=4)}, altitude_thrust)

**Look and think**
- The bouncing has died down. But the heavy drone stands about 12 cm below the target.
  A brake only stops motion; it cannot supply the missing force.
- Try `kd` = 2, 4, 8. Too small? Too big?

In [ ]:
compare({"kd = 2": PID(kp=8, kd=2), "kd = 4": PID(kp=8, kd=4), "kd = 8": PID(kp=8, kd=8)}, altitude_thrust)

## 4. I control: make up the missing force

Think of **a jar that never leaks**. While the drone is below the target, **keep pouring a little water** into the jar.
The more water, the harder it holds the drone up. The jar never leaks, so when the drone arrives and the error becomes 0, pouring just stops,
and the water already poured stays, holding up the missing weight.
In code this means adding up the error over time (the integral). While a sag remains it keeps growing, until it makes up exactly the missing force.

$$acc = k_p\,e + k_i \textstyle\sum e\,dt + k_d\,\dot e$$

✍️ Add `error × dt` to the integral, and add I to the output.

In [ ]:
class PID:
    def __init__(self, kp, ki=0.0, kd=0.0, i_band=None):
        self.kp, self.ki, self.kd = kp, ki, kd
        self.i_band = i_band
        self.integral = 0.0
        self.prev_error = None

    def update(self, error, dt):
        self.integral += ___  # ✍️
        if self.prev_error is None:
            derivative = 0.0
        else:
            derivative = (error - self.prev_error) / dt
        self.prev_error = error
        return ___  # ✍️


compare({"PD": PID(kp=8, kd=4), "PID (ki 8)": PID(kp=8, ki=8, kd=4)}, altitude_thrust)

**Look and think**
- The sag is gone. But now it passes 1.5 m by a lot. Why?
- During the 1 m climb a large error remained, and the integral piled up meanwhile (too much water was poured).
  Even after arriving, the pile keeps pushing, so it overshoots. Like cranking the shower to max and getting scalded.
  This is called integral windup.

## 5. Pause the integral when far

The fix is simple: **integrate only near the target (while the error is smaller than `i_band`, the integral band).**
Far away, P and D push hard enough, so I only needs to handle the last few cm.

✍️ Put a condition in front of the integral line. If `i_band` is `None`, always integrate.

In [ ]:
class PID:
    def __init__(self, kp, ki=0.0, kd=0.0, i_band=None):
        self.kp, self.ki, self.kd = kp, ki, kd
        self.i_band = i_band
        self.integral = 0.0
        self.prev_error = None

    def update(self, error, dt):
        if ___:  # ✍️ only while |error| < i_band (always if i_band is None)
            self.integral += error * dt
        if self.prev_error is None:
            derivative = 0.0
        else:
            derivative = (error - self.prev_error) / dt
        self.prev_error = error
        return self.kp * error + self.ki * self.integral + self.kd * derivative


compare({"PID": PID(kp=8, ki=8, kd=4), "PID + integrate only when close": PID(kp=8, ki=8, kd=4, i_band=0.15)}, altitude_thrust)

## 6. Mission: pass on both drones with my gains

Three criteria: **overshoot < 10%**, **2% settling time < 2 s**, **last-second error < 2 cm**.
Both the right-mass drone and the 10% heavy drone must pass (all six boxes green).

Order: raise `kp` to go faster → `kd` to calm the bouncing → `ki` to remove the remaining sag.

✍️ Change the numbers and run it again.

In [ ]:
MY_GAINS = dict(kp=___, ki=___, kd=___, i_band=0.15)  # ✍️

mission(PID, MY_GAINS, altitude_thrust)

Change the gains with the sliders: the moment you release one, it flies again.

In [ ]:
tune(PID, altitude_thrust)

## 7. Live view: change values while it flies

So far we computed a whole 10 s flight at once and then plotted it. A real drone is flying right now,
and we change values while watching it. The cell below keeps the drone flying **at 1× speed** (up to 2 minutes; ■ Stop ends it).

Things to try while it flies:
- Move the **target height** up and down: a step test by hand
- Bring **kd** near 0: the bouncing grows at once. Raise it again and it calms down
- Set **ki** to 0: the 10% heavy drone stops a little below the target

The middle plot splits the PID output into its three terms. You can also see D spike the moment you change the target.
Add `window=True` for a 3D window too.

In [ ]:
live(PID, altitude_thrust, **MY_GAINS)

### PlotJuggler (optional)

The plotting tool real drone teams use to look at flight logs. Just drag a value from the list on the left onto a plot,
and after stopping you can rewind. Open the window with the cell below, press **Yes → OK**, then re-run the live-flight cell above.
Today's layout (height, P·I·D, thrust) is preloaded.

In [ ]:
plotjuggler("week01")

## Recap

| What we used today | In one line |
|---|---|
| `hover_thrust` | To hang still, T = mg. Not enough on its own |
| P | Pushes by the error. Bounces and sags |
| D | Approaching fast, pushes less early. Stops the bouncing |
| I | Adds up the remaining error. Removes the sag |
| Integrate only when close | Stops I piling up and overshooting |
| Live flight | Change gains in flight and the plot changes at once |

Next session uses the same PID sideways (x, y). A drone must tilt to move sideways.